# Fine-tune Medieval YOLO on Cambridge Manuscript Annotations

Builds a YOLO-format dataset from the existing annotations, then fine-tunes
`biglam/medieval-manuscript-yolov11` on three classes:

| ID | Class | Description |
|---|---|---|
| 0 | `DropCapitalZone` | Decorated / drop capital letters |
| 1 | `GraphicZone` | Illustrations, large decorations |
| 2 | `ParagraphMark` | Paragraph marks and similar glyphs |

**Input**
- Images: `data/all_images_cropped/*.jpg`
- Labels: `data/yolo_finetune/labels/*.txt`  (polygon format: `class x1 y1 x2 y2 x3 y3 x4 y4`)

**Output**
- Dataset: `data/yolo_dataset/` (YOLO bbox layout + `data.yaml`)
- Weights: `runs/detect/<run_name>/weights/best.pt`

In [ ]:
import sys
import random
import shutil
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
IMAGES_DIR  = PROJECT_ROOT / "data" / "all_images_cropped"
LABELS_DIR  = PROJECT_ROOT / "data" / "yolo_finetune" / "labels"
DATASET_DIR = PROJECT_ROOT / "data" / "yolo_dataset"

# ── Classes ───────────────────────────────────────────────────────────────────
CLASS_NAMES = ["DropCapitalZone", "GraphicZone", "ParagraphMark"]

# ── Train / val split ─────────────────────────────────────────────────────────
VAL_FRACTION = 0.20
RANDOM_SEED  = 42

# ── Training ──────────────────────────────────────────────────────────────────
MODEL_SIZE  = "x"       # 'n' | 's' | 'm' | 'l' | 'x'
EPOCHS      = 50
IMGSZ       = 960       # divisible by 32; manuscript pages are tall
BATCH       = 4         # lower to 2 if MPS runs out of memory
LR0         = 1e-4      # low LR for fine-tuning from pretrained weights
RUN_NAME    = "cambridge-medieval-v1"

## 1 — Inspect annotations

In [ ]:
label_files = sorted(LABELS_DIR.glob("*.txt"))
print(f"Annotated images : {len(label_files)}")

class_counts = Counter()
missing_images = []

for lf in label_files:
    img = IMAGES_DIR / (lf.stem + ".jpg")
    if not img.is_file():
        missing_images.append(lf.stem)
        continue
    for line in lf.read_text().splitlines():
        parts = line.strip().split()
        if parts:
            class_counts[int(parts[0])] += 1

print(f"Missing images   : {len(missing_images)}")
print("\nClass distribution:")
for cls_id, name in enumerate(CLASS_NAMES):
    print(f"  {cls_id}  {name:<20} {class_counts[cls_id]:>4} instances")

## 2 — Build dataset

Splits annotated images into train/val, copies images, and converts polygon
labels to YOLO detection format (`class x_center y_center width height`).

In [ ]:
def polygon_to_bbox(coords: list[float]) -> tuple[float, float, float, float]:
    """Convert a 4-point polygon (8 values) to YOLO bbox (x_center, y_center, w, h)."""
    xs = coords[0::2]
    ys = coords[1::2]
    x_center = (min(xs) + max(xs)) / 2
    y_center = (min(ys) + max(ys)) / 2
    width    = max(xs) - min(xs)
    height   = max(ys) - min(ys)
    return x_center, y_center, width, height


def convert_label_file(src: Path, dst: Path) -> None:
    lines_out = []
    for line in src.read_text().splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        cls_id = parts[0]
        coords = list(map(float, parts[1:]))
        xc, yc, w, h = polygon_to_bbox(coords)
        lines_out.append(f"{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
    dst.write_text("\n".join(lines_out))

In [ ]:
# Only use annotated images that have a matching image file
valid = [lf for lf in label_files if (IMAGES_DIR / (lf.stem + ".jpg")).is_file()]

random.seed(RANDOM_SEED)
random.shuffle(valid)
n_val   = max(1, int(len(valid) * VAL_FRACTION))
val_set = set(lf.stem for lf in valid[:n_val])

print(f"Train: {len(valid) - n_val}  |  Val: {n_val}")

# Build directory tree
for split in ("train", "val"):
    (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

for lf in valid:
    split = "val" if lf.stem in val_set else "train"
    shutil.copy(IMAGES_DIR / (lf.stem + ".jpg"), DATASET_DIR / "images" / split / (lf.stem + ".jpg"))
    convert_label_file(lf, DATASET_DIR / "labels" / split / lf.name)

print(f"Dataset written to {DATASET_DIR.resolve()}")

## 3 — Write data.yaml

In [ ]:
names_block = "\n".join(f"  {i}: {name}" for i, name in enumerate(CLASS_NAMES))
yaml_content = f"""path: {DATASET_DIR.resolve()}
train: images/train
val:   images/val

nc: {len(CLASS_NAMES)}
names:\n{names_block}
"""

yaml_path = DATASET_DIR / "data.yaml"
yaml_path.write_text(yaml_content)
print(yaml_content)

## 4 — Fine-tune

In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

ckpt = hf_hub_download(
    repo_id="biglam/medieval-manuscript-yolov11",
    filename=f"medieval-yolov11{MODEL_SIZE}.pt",
)
print(f"Checkpoint: {ckpt}")

In [ ]:
model = YOLO(ckpt)

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    lr0=LR0,
    device="mps",
    name=RUN_NAME,
    exist_ok=True,
)

best_weights = Path(results.save_dir) / "weights" / "best.pt"
print(f"\nBest weights: {best_weights}")

## 5 — Validate

In [ ]:
fine_tuned = YOLO(str(best_weights))
metrics = fine_tuned.val(data=str(yaml_path), device="mps")

print(f"mAP50      : {metrics.box.map50:.3f}")
print(f"mAP50-95   : {metrics.box.map:.3f}")
print("Per-class mAP50:")
for name, ap in zip(CLASS_NAMES, metrics.box.ap50):
    print(f"  {name:<20} {ap:.3f}")